# 1부 자습 노트북 — 데이터 마이닝과 pandas 복습

본 노트북은 *직접 실행하며* 학습하는 자료이다. 셀을 위에서 아래로 *차례로 실행*하면 1부 이론 교재(`part1_data_mining_HARD_이론.md`)의 핵심 코드를 모두 돌려볼 수 있다.

**구성**: 8개 장 + 종합 실습. 각 장에서 *읽기(markdown) → 실행(code) → 확인*의 흐름이다.

**데이터셋**:
- **Ames** (2,930 × 82) — 강사 시범 (Ping)
- **Cereals** (77 × 16) — 학생 실습 (Pong)

## 환경 준비

Colab에서는 `koreanize-matplotlib`만 설치하면 된다. 로컬 Jupyter도 마찬가지다.

In [ ]:
# Colab 등에서 처음 한 번만 실행
# !pip install koreanize-matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["axes.unicode_minus"] = False
print("환경 준비 완료")

## 데이터 로딩

두 데이터셋을 GitHub raw URL에서 직접 받는다. 네트워크 실패 시 합성 데이터로 폴백.

In [ ]:
URL_AMES   = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AmesHousing.csv"
URL_CEREAL = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/Cereals.csv"

try:
    ames_raw   = pd.read_csv(URL_AMES)
    cereal_raw = pd.read_csv(URL_CEREAL)
    print(f"Ames:    {ames_raw.shape}")
    print(f"Cereals: {cereal_raw.shape}")
except Exception:
    # 폴백 — 합성 데이터
    rng = np.random.default_rng(42)
    ames_raw = pd.DataFrame({
        "Overall Qual": rng.integers(1, 11, 500),
        "Gr Liv Area":  rng.integers(500, 4000, 500),
        "SalePrice":    rng.integers(50000, 500000, 500),
        "Year Built":   rng.integers(1900, 2010, 500),
        "1st Flr SF":   rng.integers(400, 2000, 500),
        "2nd Flr SF":   rng.integers(0, 1500, 500),
        "Total Bsmt SF": rng.integers(0, 1500, 500),
        "Pool Area":    rng.integers(0, 100, 500),
    })
    cereal_raw = pd.DataFrame({
        "name":     [f"Cereal_{i}" for i in range(77)],
        "mfr":      rng.choice(list("KGNQACR"), 77),
        "type":     rng.choice(["C", "H"], 77),
        "calories": rng.integers(50, 160, 77),
        "protein":  rng.integers(1, 7, 77),
        "fat":      rng.integers(0, 6, 77),
        "sodium":   rng.integers(0, 320, 77),
        "fiber":    rng.uniform(0, 15, 77).round(1),
        "carbo":    rng.uniform(5, 25, 77).round(1),
        "sugars":   rng.integers(0, 16, 77),
        "potass":   rng.integers(20, 320, 77),
        "rating":   rng.uniform(18, 95, 77).round(2),
    })
    print(f"폴백 사용 — Ames: {ames_raw.shape}, Cereals: {cereal_raw.shape}")

---
## 0장 데이터 마이닝의 정신 — 모델 학습 전의 80%

머신러닝 프로젝트의 시간 분배는 *전처리 80%, 모델링 20%*이다. 모델은 *입력 데이터의 품질을 넘어설 수 없기* 때문이다.

본 부에서 다룰 8개 영역을 한 표로 확인한다.

In [ ]:
import pandas as pd

영역 = pd.DataFrame({
    "장": ["1장", "2장", "3장", "4장", "5장", "6장", "7장", "8장"],
    "영역": ["데이터 로딩과 첫 탐색", "변수 타입의 두 세계",
            "결측치와 이상치", "시각화",
            "변수 변환", "범주형 인코딩",
            "그룹별 집계", "특성 공학"],
    "핵심 도구": ["read_csv, info, describe", "dtypes, select_dtypes",
                  "isnull, fillna, IQR", "matplotlib + seaborn",
                  "np.log1p, StandardScaler", "pd.get_dummies, LabelEncoder",
                  "groupby, agg, transform", "도메인 지식 + 변수 조합"]
})
print(영역.to_string(index=False))

---
## 1장 데이터 로딩과 첫 탐색

데이터를 받은 직후 *반드시* 해야 할 5가지 점검이 있다.

### 1.1 shape — 행과 열의 개수

In [ ]:
print(f"Ames    shape: {ames_raw.shape}")
print(f"Cereals shape: {cereal_raw.shape}")

### 1.2 head — 데이터가 어떻게 생겼나

In [ ]:
ames_raw.head(3)

In [ ]:
cereal_raw.head(3)

### 1.3 info — 각 열의 타입과 결측

In [ ]:
cereal_raw.info()

### 1.4 describe — 수치형 열의 요약 통계

In [ ]:
cereal_raw.describe().round(2)

### 1.5 함정 — 결측치 있는 정수 열이 float64로 변함

pandas의 `int64`는 결측치(NaN)를 표현할 수 없다. 결측이 하나라도 있으면 자동으로 `float64`로 변환된다.

In [ ]:
# Garage Yr Blt는 결측이 있으므로 float64 (지하 차고가 없는 집의 정보 없음)
if "Year Built" in ames_raw.columns and "Garage Yr Blt" in ames_raw.columns:
    print(f"Year Built dtype:    {ames_raw['Year Built'].dtype}     (결측 {ames_raw['Year Built'].isnull().sum()}건)")
    print(f"Garage Yr Blt dtype: {ames_raw['Garage Yr Blt'].dtype}  (결측 {ames_raw['Garage Yr Blt'].isnull().sum()}건)")

---
## 2장 변수 타입의 두 세계 — 수치형과 범주형

pandas의 모든 변수는 *근본적으로 두 세계*로 나뉜다.
- **수치형**: 값의 크기에 의미가 있음. 평균·표준편차 자연스러움.
- **범주형**: 값은 라벨일 뿐. 평균을 계산할 수도 없음.

In [ ]:
print("Ames 타입 분포:")
print(ames_raw.dtypes.value_counts())
print()
print("Cereals 타입 분포:")
print(cereal_raw.dtypes.value_counts())

### 2.1 select_dtypes로 타입별 열 추출

In [ ]:
# 수치형만
num_cols = ames_raw.select_dtypes(include="number").columns
print(f"Ames 수치형: {len(num_cols)}개")
print(f"  예시: {num_cols[:5].tolist()}")

print()
# 범주형(문자열)만
cat_cols = ames_raw.select_dtypes(include="object").columns
print(f"Ames 범주형: {len(cat_cols)}개")
print(f"  예시: {cat_cols[:5].tolist()}")

### 2.2 함정 — 위장된 범주형

`MS SubClass`는 *숫자로 저장*되어 있지만 의미상 *주택 유형의 코드*다. 60이 20의 3배가 아니다. `astype(str)`로 명시적 변환.

In [ ]:
if "MS SubClass" in ames_raw.columns:
    print(f"변환 전 dtype: {ames_raw['MS SubClass'].dtype}")
    print(f"유일값: {sorted(ames_raw['MS SubClass'].unique())[:8]} ...")
    print()
    ames_raw["MS SubClass"] = ames_raw["MS SubClass"].astype(str)
    print(f"변환 후 dtype: {ames_raw['MS SubClass'].dtype}")
    print(f"value_counts 상위 5개:")
    print(ames_raw["MS SubClass"].value_counts().head())

---
## 3장 결측치와 이상치 — 데이터 마이닝의 두 함정

### 3.1 결측치 점검 — isnull과 sum

In [ ]:
missing = ames_raw.isnull().sum().sort_values(ascending=False).head(10)
print("Ames 결측치 상위 10개 컬럼:")
print(missing)

### 3.2 결측치 처리 — 의미 있는 결측은 "None", 진짜 결측은 중앙값

In [ ]:
ames = ames_raw.copy()

# 의미 있는 결측 ("수영장 없음", "지하 없음" 등) — "None" 라벨로
none_cols = ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
             "Garage Qual", "Garage Cond", "Garage Finish", "Garage Type",
             "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
             "BsmtFin Type 1", "BsmtFin Type 2", "Mas Vnr Type"]
for c in none_cols:
    if c in ames.columns:
        ames[c] = ames[c].fillna("None")

# 수치형 진짜 결측 — 중앙값으로
num = ames.select_dtypes("number").columns
ames[num] = ames[num].fillna(ames[num].median())

print(f"결측 남은 합계: {ames.isnull().sum().sum()}")

### 3.3 이상치 — IQR 규칙

$Q_1$, $Q_3$, IQR = $Q_3 - Q_1$. 이상치는 $[Q_1 - 1.5 \cdot \text{IQR}, Q_3 + 1.5 \cdot \text{IQR}]$ 바깥의 값.

In [ ]:
def iqr_outliers(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo = q1 - k * iqr
    hi = q3 + k * iqr
    return (series < lo) | (series > hi), (lo, hi)

mask, (lo, hi) = iqr_outliers(ames["Gr Liv Area"])
print(f"Gr Liv Area IQR 이상치: {mask.sum()}건 ({mask.mean()*100:.1f}%)")
print(f"정상 범위: {lo:.0f} ~ {hi:.0f}")

### 3.4 도메인 컷오프 — Ames의 4000 규칙

데이터 발표자 Dean De Cock이 명시한 권장 컷오프: *Gr Liv Area > 4000인 5채는 비공개 거래*이므로 제거.

In [ ]:
print(f"제거 전: {len(ames)}행")
ames = ames[ames["Gr Liv Area"] < 4000].copy()
print(f"제거 후: {len(ames)}행")

---
## 4장 시각화 — 데이터를 눈으로 보기

EDA의 4가지 질문에 답하는 시각화 4종을 차례로 그려본다.

### 4.1 (1) 히스토그램 — 한 변수의 분포

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(ames["SalePrice"] / 1000, bins=50, color="#1F3A5F", alpha=0.7, edgecolor="white")
ax.set_xlabel("SalePrice ($1000)")
ax.set_ylabel("빈도")
ax.set_title("Ames 주택 가격 분포 — 오른쪽 꼬리(skewed right)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"비대칭도(skewness): {ames['SalePrice'].skew():.3f}  (> 0이면 오른쪽 꼬리)")

### 4.2 (2) 산점도 — 두 변수의 관계

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(ames["Gr Liv Area"], ames["SalePrice"] / 1000,
           c="#1F3A5F", s=8, alpha=0.4)
ax.set_xlabel("Gr Liv Area (sq ft)")
ax.set_ylabel("SalePrice ($1000)")
ax.set_title("거실 면적 vs 판매 가격 — 양의 상관")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 (3) 그룹별 박스플롯 — 범주별 분포 비교

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=ames, x="Overall Qual", y="SalePrice", ax=ax)
ax.set_title("품질 등급별 가격 — 단조 증가 패턴")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### 4.4 (4) 히트맵 — 다변수 상관행렬

Cereals 영양 성분 사이의 상관계수.

In [ ]:
import seaborn as sns

cols = ["calories", "protein", "fat", "sugars", "fiber", "rating"]
corr = cereal_raw[cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Cereals 영양 상관행렬 — sugars↔rating 강한 음의 상관")
plt.tight_layout()
plt.show()

---
## 5장 변수 변환 — 로그·표준화

오른쪽 꼬리 분포를 *대칭에 가깝게* 만드는 로그 변환과, *평균 0·표준편차 1*로 맞추는 표준화를 차례로 적용한다.

### 5.1 로그 변환 효과 측정

In [ ]:
raw_skew = ames["SalePrice"].skew()
log_skew = np.log1p(ames["SalePrice"]).skew()

print(f"원본 비대칭도:    {raw_skew:+.3f}")
print(f"로그 변환 후:    {log_skew:+.3f}")
print(f"개선 폭:        {abs(raw_skew - log_skew):.3f}")

### 5.2 로그 변환 비교 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(ames["SalePrice"] / 1000, bins=50, color="#1F3A5F", alpha=0.7)
axes[0].set_xlabel("SalePrice ($1000)")
axes[0].set_title(f"원본 — skewness {raw_skew:.2f}")
axes[0].grid(alpha=0.3)

axes[1].hist(np.log1p(ames["SalePrice"]), bins=50, color="#5A8C5A", alpha=0.7)
axes[1].set_xlabel("log(1 + SalePrice)")
axes[1].set_title(f"로그 변환 후 — skewness {log_skew:.2f}")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.3 표준화 — StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ames.select_dtypes("number").columns
scaler = StandardScaler()
X_scaled = scaler.fit_transform(ames[num_cols])

print(f"변환 후 평균:    {X_scaled.mean(axis=0)[:3]}")
print(f"변환 후 표준편차: {X_scaled.std(axis=0)[:3]}")
print()
print("거의 0, 거의 1로 맞춰졌다. 이제 어떤 모델에도 안전하게 넣을 수 있다.")

---
## 6장 범주형 인코딩 — 라벨과 원-핫

### 6.1 라벨 인코딩 — 한 컬럼, 정수 코드

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
mfr_encoded = le.fit_transform(cereal_raw["mfr"])

print(f"원본 mfr: {cereal_raw['mfr'].unique().tolist()}")
print(f"인코딩:   {sorted(set(mfr_encoded))}")
print(f"매핑:     {dict(zip(le.classes_, range(len(le.classes_))))}")

### 6.2 원-핫 인코딩 — K개 컬럼

`pd.get_dummies`로 한 줄.

In [ ]:
mfr_onehot = pd.get_dummies(cereal_raw["mfr"], prefix="mfr")
print(f"원-핫 shape: {mfr_onehot.shape}")
print()
print(mfr_onehot.head())

### 6.3 drop_first로 다중공선성 회피 (선형회귀용)

In [ ]:
mfr_safe = pd.get_dummies(cereal_raw["mfr"], prefix="mfr", drop_first=True)
print(f"K개 범주: {len(cereal_raw['mfr'].unique())}")
print(f"drop_first=True 후: {mfr_safe.shape[1]}개 컬럼 (K-1)")

---
## 7장 그룹별 집계 — groupby

Split-Apply-Combine 세 단계.

### 7.1 가장 단순한 groupby

In [ ]:
mean_cal = cereal_raw.groupby("mfr")["calories"].mean().sort_values(ascending=False)
print("제조사별 평균 칼로리:")
print(mean_cal.round(1))

### 7.2 agg — 여러 집계 함수 동시

In [ ]:
agg = cereal_raw.groupby("mfr")["calories"].agg(["mean", "std", "min", "max", "count"])
print(agg.round(1))

### 7.3 transform — 그룹별 통계를 원본 행에 붙이기

In [ ]:
# Ames에서 동네별 평균 가격을 모든 주택에 붙이기
if "Neighborhood" in ames.columns:
    ames["NbhdMeanPrice"] = ames.groupby("Neighborhood")["SalePrice"].transform("mean")
    print(ames[["Neighborhood", "SalePrice", "NbhdMeanPrice"]].head(8))
    print()
    print(f"shape 확인: 원본 {ames.shape[0]}행, transform 결과 길이 {len(ames['NbhdMeanPrice'])}행")

---
## 8장 특성 공학 — 도메인 지식을 변수로

*세상의 지식*을 *변수의 형태*로 모델에 전달한다.

### 8.1 Total SF — 세 면적 변수를 합쳐 더 강한 변수

In [ ]:
if all(c in ames.columns for c in ["1st Flr SF", "2nd Flr SF", "Total Bsmt SF"]):
    ames["Total SF"] = ames["1st Flr SF"] + ames["2nd Flr SF"] + ames["Total Bsmt SF"]
    
    # 상관계수 비교
    original = ames[["1st Flr SF", "2nd Flr SF", "Total Bsmt SF", "SalePrice"]].corr()["SalePrice"][:-1]
    new      = ames[["Total SF", "SalePrice"]].corr().iloc[0, 1]
    
    print("원본 세 변수의 SalePrice 상관계수:")
    print(original.round(3))
    print()
    print(f"새로 만든 Total SF의 상관계수: {new:.3f}")
    print(f"\n→ 도메인 지식('가격은 총 면적에 비례')이 더 강한 신호를 만들었다.")

### 8.2 시간 차이 변수 — HouseAge

In [ ]:
if "Yr Sold" in ames.columns and "Year Built" in ames.columns:
    ames["HouseAge"] = ames["Yr Sold"] - ames["Year Built"]
    ames["YearsSinceRemodel"] = ames["Yr Sold"] - ames["Year Remod/Add"]
    
    print(f"HouseAge 범위:        {ames['HouseAge'].min()} ~ {ames['HouseAge'].max()}년")
    print(f"YearsSinceRemodel 범위: {ames['YearsSinceRemodel'].min()} ~ {ames['YearsSinceRemodel'].max()}년")
    print()
    print(f"HouseAge          SalePrice 상관: {ames[['HouseAge', 'SalePrice']].corr().iloc[0,1]:.3f}")
    print(f"YearsSinceRemodel SalePrice 상관: {ames[['YearsSinceRemodel', 'SalePrice']].corr().iloc[0,1]:.3f}")

### 8.3 이진 플래그 변수

In [ ]:
if "Pool Area" in ames.columns:
    ames["HasPool"] = (ames["Pool Area"] > 0).astype(int)
    print(f"수영장 있는 집: {ames['HasPool'].sum()}채 ({ames['HasPool'].mean()*100:.2f}%)")

if "Garage Area" in ames.columns:
    ames["HasGarage"] = (ames["Garage Area"] > 0).astype(int)
    print(f"차고 있는 집:  {ames['HasGarage'].sum()}채 ({ames['HasGarage'].mean()*100:.2f}%)")

### 8.4 Cereals에서 영양 점수 만들기

In [ ]:
cereal = cereal_raw.copy()
num_c = cereal.select_dtypes("number").columns
cereal[num_c] = cereal[num_c].fillna(cereal[num_c].median())

# 도메인 지식: 단백질·섬유 좋음, 지방·설탕 나쁨
cereal["NutritionScore"] = (
    2.0 * cereal["protein"]
    + 1.5 * cereal["fiber"]
    - 1.0 * cereal["fat"]
    - 1.0 * cereal["sugars"]
)

r = cereal[["NutritionScore", "rating"]].corr().iloc[0, 1]
print(f"NutritionScore와 rating의 상관계수: {r:+.3f}")
print("도메인 지식이 강한 예측 신호를 만들었다.")

---
## 종합 실습 — 표준 전처리 파이프라인

1부에서 다룬 모든 기법을 *한 함수*로 묶는다. 2~5부의 모든 모델링이 이 함수의 출력을 입력으로 받는다.

In [ ]:
def prepare_ames(df_in):
    df = df_in.copy()
    
    # 1. ID 컬럼 제거
    df = df.drop(columns=[c for c in ["Order", "PID"] if c in df.columns])
    
    # 2. 의미 있는 결측을 "None"으로
    none_cols = ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
                 "Garage Qual", "Garage Cond", "Garage Finish", "Garage Type",
                 "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
                 "BsmtFin Type 1", "BsmtFin Type 2", "Mas Vnr Type"]
    for c in none_cols:
        if c in df.columns:
            df[c] = df[c].fillna("None")
    
    # 3. 수치형 결측을 중앙값으로
    num_cols = df.select_dtypes("number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    
    # 4. 이상치 제거 (Gr Liv Area > 4000)
    if "Gr Liv Area" in df.columns:
        df = df[df["Gr Liv Area"] < 4000].copy()
    
    # 5. 특성 공학 (Total SF)
    if all(c in df.columns for c in ["1st Flr SF", "2nd Flr SF", "Total Bsmt SF"]):
        df["Total SF"] = df["1st Flr SF"] + df["2nd Flr SF"] + df["Total Bsmt SF"]
    
    # 6. 위장된 범주형 변환
    if "MS SubClass" in df.columns:
        df["MS SubClass"] = df["MS SubClass"].astype(str)
    
    return df

ames_clean = prepare_ames(ames_raw)
print(f"전처리 결과: {ames_clean.shape}")
print(f"결측 합계:   {ames_clean.isnull().sum().sum()}")
print()
print("이 함수가 2~5부의 모든 모델링의 출발점이다.")

---
## 마무리

본 노트북에서 *직접 실행*한 8가지 도구를 한 표로 정리한다.

| 영역 | 핵심 도구 |
|---|---|
| 1장 로딩과 첫 탐색 | `read_csv`, `shape`, `head`, `info`, `describe` |
| 2장 변수 타입 | `dtypes`, `select_dtypes`, `astype` |
| 3장 결측치와 이상치 | `fillna`, IQR 규칙, 도메인 컷오프 |
| 4장 시각화 | matplotlib + seaborn 4종 |
| 5장 변수 변환 | `np.log1p`, `StandardScaler` |
| 6장 인코딩 | `LabelEncoder`, `pd.get_dummies(drop_first=True)` |
| 7장 그룹별 집계 | `groupby`, `agg`, `transform` |
| 8장 특성 공학 | Total SF, HouseAge, NutritionScore |

### 다음 단계

2부로 이동하여 **결정 트리 심화**(`part2_tree_advanced_HARD_자습노트북.ipynb`)를 자습한다. 본 부에서 만든 *깨끗한 데이터*를 *결정 트리*에 넣어 *어떤 패턴이 잡히는지* 본격적으로 분석한다.